# Derive models of drone

Do all imports.

In [30]:
import sympy as sym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import secrets
import ae353_drone

# Suppress the use of scientific notation when printing small numbers
np.set_printoptions(suppress=True)

## Dynamic model

Define physical parameters.

In [31]:
params = {
    'm': 0.5,
    'Jx': 0.0023,
    'Jy': 0.0023,
    'Jz': 0.0040,
    'l': 0.175,
    'g': 9.81,
}

Derive the equations of motion:

In [32]:
# components of position (meters)
p_x, p_y, p_z = sym.symbols('p_x, p_y, p_z')

# yaw, pitch, roll angles (radians)
psi, theta, phi = sym.symbols('psi, theta, phi')

# components of linear velocity (meters / second)
v_x, v_y, v_z = sym.symbols('v_x, v_y, v_z')
v_in_body = sym.Matrix([v_x, v_y, v_z])

# components of angular velocity (radians / second)
w_x, w_y, w_z = sym.symbols('w_x, w_y, w_z')
w_in_body = sym.Matrix([w_x, w_y, w_z])

# components of net rotor torque
tau_x, tau_y, tau_z = sym.symbols('tau_x, tau_y, tau_z')

# net rotor force
f_z = sym.symbols('f_z')

# parameters
m = sym.nsimplify(params['m'])
Jx = sym.nsimplify(params['Jx'])
Jy = sym.nsimplify(params['Jy'])
Jz = sym.nsimplify(params['Jz'])
l = sym.nsimplify(params['l'])
g = sym.nsimplify(params['g'])
J = sym.diag(Jx, Jy, Jz)

# rotation matrices
Rz = sym.Matrix([[sym.cos(psi), -sym.sin(psi), 0], [sym.sin(psi), sym.cos(psi), 0], [0, 0, 1]])
Ry = sym.Matrix([[sym.cos(theta), 0, sym.sin(theta)], [0, 1, 0], [-sym.sin(theta), 0, sym.cos(theta)]])
Rx = sym.Matrix([[1, 0, 0], [0, sym.cos(phi), -sym.sin(phi)], [0, sym.sin(phi), sym.cos(phi)]])
R_body_in_world = Rz @ Ry @ Rx

# angular velocity to angular rates
ex = sym.Matrix([[1], [0], [0]])
ey = sym.Matrix([[0], [1], [0]])
ez = sym.Matrix([[0], [0], [1]])
M = sym.simplify(sym.Matrix.hstack((Ry @ Rx).T @ ez, Rx.T @ ey, ex).inv(), full=True)

# applied forces
f_in_body = R_body_in_world.T @ sym.Matrix([[0], [0], [-m * g]]) + sym.Matrix([[0], [0], [f_z]])

# applied torques
tau_in_body = sym.Matrix([[tau_x], [tau_y], [tau_z]])

# equations of motion
f = sym.Matrix.vstack(
    R_body_in_world @ v_in_body,
    M @ w_in_body,
    (1 / m) * (f_in_body - w_in_body.cross(m * v_in_body)),
    J.inv() @ (tau_in_body - w_in_body.cross(J @ w_in_body)),
)

f = sym.simplify(f, full=True)

The equations of motion have this form:

$$\begin{bmatrix} \dot{p}_x \\ \dot{p}_y \\ \dot{p}_z \\ \dot{\psi} \\ \dot{\theta} \\ \dot{\phi} \\ \dot{v}_x \\ \dot{v}_y \\ \dot{v}_z \\ \dot{w}_x \\ \dot{w}_y \\ \dot{w}_z \end{bmatrix} = f\left(p_x, p_y, p_z, \psi, \theta, \phi, v_x, v_y, v_z, w_x, w_y, w_z, \tau_x, \tau_y, \tau_z, f_z \right)$$

Here is the function $f$:

In [33]:
f

Matrix([
[v_x*cos(psi)*cos(theta) + v_y*(sin(phi)*sin(theta)*cos(psi) - sin(psi)*cos(phi)) + v_z*(sin(phi)*sin(psi) + sin(theta)*cos(phi)*cos(psi))],
[v_x*sin(psi)*cos(theta) + v_y*(sin(phi)*sin(psi)*sin(theta) + cos(phi)*cos(psi)) - v_z*(sin(phi)*cos(psi) - sin(psi)*sin(theta)*cos(phi))],
[                                                                      -v_x*sin(theta) + v_y*sin(phi)*cos(theta) + v_z*cos(phi)*cos(theta)],
[                                                                                                 (w_y*sin(phi) + w_z*cos(phi))/cos(theta)],
[                                                                                                              w_y*cos(phi) - w_z*sin(phi)],
[                                                                                  w_x + w_y*sin(phi)*tan(theta) + w_z*cos(phi)*tan(theta)],
[                                                                                                   v_y*w_z - v_z*w_y + 981*sin(theta)/100],
[   

## Sensor model

Define the sensor model.

In [34]:
# Position of drone in world frame
p_in_world = sym.Matrix([p_x, p_y, p_z])

# Position of markers in body frame
a_in_body = sym.Matrix([l, 0, 0])  # <-- marker on front rotor
b_in_body = sym.Matrix([-l, 0, 0]) # <-- marker on back rotor

# Position of markers in world frame
a_in_world = p_in_world + R_body_in_world @ a_in_body
b_in_world = p_in_world + R_body_in_world @ b_in_body

# Sensor model
g = sym.simplify(sym.Matrix.vstack(a_in_world, b_in_world))

The sensor model has this form:

$$o = g(p_x, p_y, p_z, \psi, \theta, \phi)$$

Here is the function $g$:

In [35]:
g

Matrix([
[p_x + 7*cos(psi)*cos(theta)/40],
[p_y + 7*sin(psi)*cos(theta)/40],
[         p_z - 7*sin(theta)/40],
[p_x - 7*cos(psi)*cos(theta)/40],
[p_y - 7*sin(psi)*cos(theta)/40],
[         p_z + 7*sin(theta)/40]])

## Model-based controller and observer design

### Linearization

Define the state and input of the nonlinear system:

$$
m = \begin{bmatrix} p_x \\ p_y \\ p_z \\ \psi \\ \theta \\ \phi \\ v_x \\ v_y \\ v_z \\ w_x \\ w_y \\ w_z \end{bmatrix}
\qquad\qquad
n = \begin{bmatrix} \tau_x \\ \tau_y \\ \tau_z \\ f_z \end{bmatrix}.
$$

In [36]:
# All elements of the state (of the nonlinear system) as symbolic variables
m = [p_x, p_y, p_z, psi, theta, phi,
     v_x, v_y, v_z, w_x, w_y, w_z]


# All elements of the input (of the nonlinear system) as symbolic variables
n = [tau_x, tau_y, tau_z, f_z]

Choose an equilibrium point $m_e$ and $n_e$ that corresponds to hover at the origin with a zero yaw angle and that satisfies

$$ 0 = f(m_e, n_e).$$

In [37]:
# f as a lambda function that can be evaluated for different choices of m and n
f_num = sym.lambdify(m + n, f)

# Equilibrium value of all elements of the state (of the nonlinear system)
m_e = [0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0]
# Equilibrium value of all elements of the input (of the nonlinear system)
n_e = [0, 0, 0, .5 * 9.81 ]

# Check if m_e, n_e really is an equilibrium point.
assert(np.isclose(f_num(*(m_e + n_e)), 0).all())

Linearize the nonlinear dynamic model

$$ \dot{m} = f(m, n) $$

to produce a linear dynamic model

$$ \dot{x} = A x + B u $$

where

$$
A = \frac{\partial f}{\partial m}\biggr\rvert_{(m_e, n_e)}
\qquad\qquad
B = \frac{\partial f}{\partial n}\biggr\rvert_{(m_e, n_e)}
$$

and

$$
x = m - m_e
\qquad\qquad
u = n - n_e.
$$

In [38]:
# Create lambda functions to evaluate the Jacobian of f with respect to m and n at different values of m_e and n_e.
A_num = sym.lambdify(m + n, f.jacobian(m))
B_num = sym.lambdify(m + n, f.jacobian(n))

# Evaluate these lambda functions at the values of m_e and n_e

A = A_num(*(m_e + n_e))
B = B_num(*(m_e + n_e))

Linearize the nonlinear sensor model

$$ o = g(m, n) $$

to produce a linear sensor model

$$ y = Cx + Du $$

where

$$
C = \frac{\partial f}{\partial m}\biggr\rvert_{(m_e, n_e)}
\qquad\qquad
D = \frac{\partial f}{\partial n}\biggr\rvert_{(m_e, n_e)}
$$

and

$$
x = m - m_e
\qquad\qquad
u = n - n_e
\qquad\qquad
y = o - g(m_e, n_e).
$$

(You may find that $D = 0$ and so that part of your linear sensor model can be ignored.)

In [39]:
# Lambda functions to evaluate the Jacobian of g with
# respect to m and n at different values of m_e and n_e.
C_num = sym.lambdify(m + n, g.jacobian(m))
D_num = sym.lambdify(m + n, g.jacobian(n))

# Evaluate these lambda functions at these values of m_e and n_e.
C = C_num(*(m_e + n_e))
D = D_num(*(m_e + n_e))

# g as a lambda function that can be evaluated for different choices of m and n
g_num = sym.lambdify(m + n, g)

# Find the sensor measurements that you would expect to see at equilibrium
o_e = g_num(*(m_e + n_e)).reshape(-1)

Make sure that `m_e`, `n_e`, and `o_e` are all 1D numpy arrays. (Until now, it was likely easier to work with `m_e` and `n_e` as lists instead of as arrays, for example.)

In [40]:
m_e = np.array(m_e)
n_e = np.array(n_e)
o_e = np.array(o_e)

assert(m_e.ndim == 1)
assert(n_e.ndim == 1)
assert(o_e.ndim == 1)

### Controller design

Check that the system is controllable.

In [41]:
# Verify that the controllability matrix is full rank.
def w(A, B):
    n = A.shape[0]
    W = B
    for i in range(1, n):
        W = np.hstack((W, np.linalg.matrix_power(A, i) @ B))
    return W
assert(np.linalg.matrix_rank(w(A, B)) == A.shape[0])

Define a function that solves the continuous-time, infinite-horizon LQR problem.

(You can find an implementation of this function at the [aerospace control systems reference page](https://aero.refpages.org/control/) at the end of the "Optimal controllers" section.)

In [42]:
from scipy.linalg import solve_continuous_are
def lqr(A, B, Q, R):
    P = solve_continuous_are(A, B, Q, R)
    return np.linalg.inv(R) @ B.T @ P # = K

Choose weights $Q_c$ and $R_c$. Remember that $Q_c$ is a diagonal, square matrix of size $n_\text{x} \times n_\text{x}$ where $n_x$ is the number of states. Similarly, $R_c$ is a diagonal, square matrix of size $n_\text{u} \times n_\text{u}$ where $n_u$ is the number of inputs.

In [43]:
Q_c = 0.1 * np.diag([10, 10, 2e4, 5, 10, 10, 50, 50, 50, 100, 100, 100])
R_c = 1e4 * np.eye(4)
R_c[3,3] = 50 # this was changed to get better altitude response

Find the gain matrix $K$ that solves the optimal controller design problem corresponding to your choice of $Q_c$ and $R_c$.

In [44]:
K = lqr(A, B, Q_c, R_c)
sym.Matrix(np.round(K, decimals=10))

Matrix([
[ 0.0, -0.01,          0.0,          0.0,          0.0, 0.1501861867,          0.0, -0.0283934849,          0.0, 0.041120025,         0.0,          0.0],
[0.01,   0.0,          0.0,          0.0, 0.1501861867,          0.0, 0.0283934849,           0.0,          0.0,         0.0, 0.041120025,          0.0],
[ 0.0,   0.0,          0.0, 0.0070710678,          0.0,          0.0,          0.0,           0.0,          0.0,         0.0,         0.0, 0.0325049003],
[ 0.0,   0.0, 6.3245553203,          0.0,          0.0,          0.0,          0.0,           0.0, 2.5346706532,         0.0,         0.0,          0.0]])

### Observer design

Check that the system is observable.

In [45]:
# Verify that the observability matrix is full rank.
def Wo(A, C):
    n = A.shape[0]
    O = C
    for i in range(1, n):
        O = np.vstack((O, C @ np.linalg.matrix_power(A, i)))
    return O
assert(np.linalg.matrix_rank(Wo(A, C)) == A.shape[0])

Choose weights $Q_o$ and $R_o$. Remember that $Q_o$ is a diagonal, square matrix of size $n_\text{y} \times n_\text{y}$ where $n_y$ is the number of outputs. Similarly, $R_o$ is a diagonal, square matrix of size $n_\text{x} \times n_\text{x}$ where $n_x$ is the number of states.

In [46]:
Q_o = 1 * np.diag([1,1,1,1,1,1])
R_o = np.diag([1,1,1,1,1,1,1,1,1,1,1,1])

Find the gain matrix $L$ that solves the optimal observer design problem corresponding to your choice of $Q_o$ and $R_o$.

(The end of the "Optimal observers" section of the [aerospace control systems reference page](https://aero.refpages.org/control/) reminds you how to use the same `lqr` function you defined nefore to solve the optimal observer design problem — the syntax is a little different than for solving the optimal controller design problem.)

In [47]:
L = lqr(A.T, C.T, R_o, Q_o).T
sym.Matrix(np.round(L, decimals=10))

Matrix([
[2.9271137918,           0.0, -0.2197681637, 2.9271137918,           0.0, 0.2197681637],
[         0.0,   2.975376019,           0.0,          0.0,   2.975376019,          0.0],
[         0.0,           0.0,  1.0986841135,          0.0,           0.0, 1.0986841135],
[         0.0,  2.1308707559,           0.0,          0.0, -2.1308707559,          0.0],
[1.2558180782,           0.0, -0.3042973804, 1.2558180782,           0.0, 0.3042973804],
[         0.0,  -1.305431739,           0.0,          0.0,  -1.305431739,          0.0],
[8.1162931959,           0.0, -1.1915335688, 8.1162931959,           0.0, 1.1915335688],
[         0.0,  8.3528624542,           0.0,          0.0,  8.3528624542,          0.0],
[         0.0,           0.0,  0.7071067812,          0.0,           0.0, 0.7071067812],
[         0.0, -0.7071067812,           0.0,          0.0, -0.7071067812,          0.0],
[0.6768313358,           0.0, -0.2046932897, 0.6768313358,           0.0, 0.2046932897],
[         0.

### Make it easy to copy/paste your design into a `Controller` implementation

Define a function that rounds all elements of a numpy array to a chosen number of digits after the decimal (there is no need to keep around too many significant figures) and converts the result to a list so it can be printed compactly.

In [48]:
def rnd(a, decimals=5):
    return np.round(a, decimals=decimals).tolist()

Print lines of text to copy/paste into a `Controller` implementation (if desired).

In [49]:
print(f'        self.A = np.array({rnd(A)})')
print(f'        self.B = np.array({rnd(B)})')
print(f'        self.C = np.array({rnd(C)})')
print(f'        self.K = np.array({rnd(K)})')
print(f'        self.L = np.array({rnd(L)})')
print(f'        self.m_e = np.array({rnd(m_e)})')
print(f'        self.n_e = np.array({rnd(n_e)})')
print(f'        self.o_e = np.array({rnd(o_e)})')

        self.A = np.array([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, -0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 9.81, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, -9.81, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
        self.B = np.array([[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 2.

# Quadrotor (aka "the drone")

The following cell can be used to select a random seed. For our testing, we chose to use random seeds to test the robustness of the drone under random conditions.

For this code submission, we chose set seeds for reproducibility.

In [50]:
# code to generate random seeds. This is optional since using seed 'None' will give random initial conditions and obstacle/ring placements.
seeds = []
for i in range(100):
    seed = secrets.randbits(32)
    seeds.append(seed)
print(seeds)
# for reproducibility:
seeds = [2812816404, 3319318006, 4134580205, 512536746, 3273172, 1482039977, 2105047229, 734853473, 1447591964, 280058207, 2403158312, 1730952578, 2965541946, 1227161350, 1712631873, 3262058154, 1832764077, 3289404194, 2054551452, 3095679145, 815448959, 2484579243, 3380000840, 3561208952, 3607607621, 1056684227, 1906033592, 430846658, 2375084387, 2523463334, 3731812540, 580132413, 2816419673, 343654134, 1550748766, 1665080722, 3782336809, 1910171075, 2082180065, 2667812483, 537464174, 2096516479, 3135564586, 530478035, 3798215386, 3920953163, 264578494, 3465300166, 4078772635, 1825318116, 3612984420, 2711104723, 388136987, 1184547647, 536264869, 177462116, 1590664855, 2831844785, 2514272229, 379307730, 1910761814, 1394130460, 668871610, 1689662277, 2968722032, 3501356906, 1613659932, 1554197767, 1558299263, 4151255549, 113833022, 1933679150, 1254406836, 592777328, 1520712260, 1695080225, 2268309066, 1626161924, 2048803010, 2582911230, 3341063493, 2364145758, 1307744386, 2358205254, 1939055455, 1870801166, 2050707370, 3308510432, 622200493, 3309875839, 789211350, 640582918, 2195910550, 873730423, 2422743589, 1682204637, 2117565206, 594633587, 205111785, 2740526439]

[70112070, 1117343792, 808817165, 875965277, 206217466, 3464121294, 2248803479, 1881519280, 563232547, 3273772303, 594936310, 1235566343, 4260648606, 2778480338, 317547965, 1122782630, 3882129498, 1045528342, 918308538, 3732180321, 3186488, 2023499108, 2713808838, 491021385, 673925590, 1744016825, 3928681084, 914440162, 2466275894, 562896717, 3361536425, 1891091955, 3499518799, 2299741674, 1958994264, 1845852926, 3969042194, 2246849608, 414934049, 1219800, 3413525897, 2180378000, 2562656308, 312453795, 2784618555, 1072372541, 1330313154, 2138839173, 1872157193, 4207986611, 2958729540, 1779397364, 3372662491, 50304337, 697940581, 3801136594, 4061779945, 3051152215, 1401634739, 1642755791, 3046412943, 2182504390, 1875655854, 1598891529, 274100200, 2814950469, 3252029987, 2465674180, 1363395965, 3205590990, 1562520776, 709214751, 3152768320, 3068997580, 2254236514, 1165728187, 115185681, 206926382, 3776585374, 62051380, 1399911685, 1983367971, 1226726128, 2377045480, 226574045, 963905031,

Create simulator with seed.

If your simulator runs much slower than real-time, then you can try adding the optional argument `display_pybullet=True` (instead of using meshcat views).

In [51]:
simulator = ae353_drone.Simulator(seed=seeds[0]) # in this case, the first seed from the list

Add a camera view. This view will be displayed in its own browser window.

In [52]:
simulator.add_view(
    'back_view',  # name of view (must be unique)
    'back',       # type of view (start, top, right, left, or back)
)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7001/static/


Define a controller for the drone.

In [53]:
import numpy as np

class Controller:
    def __init__(self):
        # System matrices and gains (assumed defined globally)
        self.A = A
        self.B = B
        self.C = C
        self.K = K
        self.L = L
        # Equilibrium (hover, level)
        self.m_e = np.zeros(12)
        self.n_e = np.array([0.0, 0.0, 0.0, 4.905])
        self.o_e = np.array([0.175, 0.0, 0.0, -0.175, 0.0, 0.0])
        self.f_ze = self.n_e[3]

        # Timing
        self.dt = 0.04  # 25 Hz

        # Obstacle repulsion (dogs / other drones)
        self.k_repel      = 0.5 # gain for real obstacles
        self.k_repel_ring = 0.2 # smaller gain for ring obstacle (tune this)
        self.r_drone = 0.3      # safe radius around drone
        self.R_far   = 4.0      # radius within which repulsion is active

        # Reference motion (low-pass waypoint tracking, base values)
        self.k_ref_pos = 2.85   # p_ref' = k_ref_pos (p_target - p_ref)
        self.v_max     = 1.0    # |v_ref| <= v_max

        # Ring targeting (near/far waypoints)
        self.vertical_offset     = 0.2
        self.horizontal_offset   = 0.85
        self.near_reached_radius = 0.75
        self.far_reached_radius  = 0.75

        # Ring geometry (for closest-point calculation)
        self.ring_radius = 1.0

        # Active ring visit state
        self.active_ring_center    = None
        self.active_ring_dir       = None
        self.active_side_sign      = 1.0
        self.ring_phase            = 'near'   # 'near' or 'far'
        self.active_ring_completed = True

        # All rings seen so far: list of dicts {'center': ..., 'dir': ...}
        self.rings = []

        # Internal reference position
        self.p_ref = None

        # Logging
        self.variables_to_log = ['xhat', 'xdes']

    def get_color(self):
        # Team Luke
        return [0.114, 0.788, 0.42]

    def reset(self, p_x, p_y, p_z, yaw):
        # State estimate
        self.xhat = np.array([
            p_x, p_y, p_z,
            yaw, 0.0, 0.0,
            0.0, 0.0, 0.0,
            0.0, 0.0, 0.0
        ])
        self.xdes = np.zeros(12)

        # Reference position
        self.p_ref = np.array([p_x, p_y, p_z], dtype=float)

        # Ring visit state
        self.active_ring_center    = None
        self.active_ring_dir       = None
        self.active_side_sign      = 1.0
        self.ring_phase            = 'near'
        self.active_ring_completed = True

        # Clear ring list
        self.rings = []

    def _register_ring(self, pos_ring, dir_ring):
        # Store each ring (center + unit normal) once.
        center = np.array(pos_ring, dtype=float)
        n = np.array(dir_ring, dtype=float)
        n_norm = np.linalg.norm(n)
        n = n / n_norm if n_norm > 1e-6 else np.array([1.0, 0.0, 0.0])

        for r in self.rings:
            if np.linalg.norm(r['center'] - center) < 1e-3:
                r['dir'] = n
                return

        self.rings.append({'center': center, 'dir': n})

    def _closest_ring_point(self, p_est):
        # Determine closest point on hoop of closest ring to p_est.
        if len(self.rings) == 0:
            return None

        best_point = None
        best_dist = np.inf

        for r in self.rings:
            c = r['center']
            n = r['dir']

            # Vector from center to drone
            q = p_est - c
            # Decompose into plane (perpendicular to n)
            d_n = np.dot(q, n)
            q_plane = q - d_n * n
            r_plane = np.linalg.norm(q_plane)

            if r_plane < 1e-6:
                # Drone is almost on the normal through the center:
                if abs(n[0]) < 0.9:
                    a = np.array([1.0, 0.0, 0.0])
                else:
                    a = np.array([0.0, 1.0, 0.0])
                q_plane_dir = np.cross(n, a)
                q_plane_dir /= np.linalg.norm(q_plane_dir)
            else:
                q_plane_dir = q_plane / r_plane

            # Closest point on ring hoop
            ring_point = c + self.ring_radius * q_plane_dir

            d = np.linalg.norm(p_est - ring_point)
            if d < best_dist:
                best_dist = d
                best_point = ring_point

        return best_point

    def _lock_new_ring_visit(self, p_est, pos_ring, dir_ring):
        # Lock in a new ring visit (center, dir, side) when no active ring is present.
        center = np.array(pos_ring, dtype=float)
        n = np.array(dir_ring, dtype=float)
        n_norm = np.linalg.norm(n)
        n = n / n_norm if n_norm > 1e-6 else np.array([1.0, 0.0, 0.0])

        v_to_ring = p_est - center
        side_sign = -1.0 if np.dot(v_to_ring, n) < 0.0 else 1.0

        self.active_ring_center    = center
        self.active_ring_dir       = n
        self.active_side_sign      = side_sign
        self.ring_phase            = 'near'
        self.active_ring_completed = False

    def _obstacle_repulsion(self, p_est, pos_list, gain):
        # Generic repulsion for a list of obstacle positions using a specified gain.
        if gain <= 0.0 or pos_list is None or len(pos_list) == 0:
            return np.zeros(3)

        h_repel = np.zeros(3)
        for obst_pos in np.atleast_2d(pos_list):
            diff = p_est - obst_pos
            dist = np.linalg.norm(diff)
            if dist < 1e-6 or dist > self.R_far:
                continue

            d_clear = dist - self.r_drone
            if d_clear <= 0.0:
                d_clear = 0.05

            denom_far = self.R_far - self.r_drone
            if denom_far <= 0.0:
                denom_far = 0.1

            mag = (1.0 / d_clear) - (1.0 / denom_far)
            if mag <= 0.0:
                continue

            h_repel += mag * (diff / dist)

        return gain * h_repel

    def _local_speed_limits(self, p_est, ring_center, pos_all_for_speed):
        # Slow speed near ring and obstacles,
        # speed up when far away
        d_ring = np.linalg.norm(p_est - ring_center)

        d_obst = np.inf
        if pos_all_for_speed is not None and len(pos_all_for_speed) > 0:
            for obst_pos in np.atleast_2d(pos_all_for_speed):
                d = np.linalg.norm(obst_pos - p_est)
                if d < d_obst:
                    d_obst = d

        v = self.v_max
        k = self.k_ref_pos
        
        if d_obst > 4.0:    # Far from everything
            v *= 2.5
            k *= 2.0
        elif d_obst > 2.5:  # Medium distance
            v *= 2.0
            k *= 1.5
        elif d_obst < 1.5:  # Very close to ring or obstacle
            v *= 0.7
            k *= 0.8
        return v, k
    def run(self, pos_markers, pos_ring, dir_ring, is_last_ring, pos_others):
        # Position estimate
        p_est = self.xhat[0:3]
        if self.p_ref is None:
            self.p_ref = p_est.copy()

        # Track rings and ring visit logic (near → far)
        self._register_ring(pos_ring, dir_ring)

        if self.active_ring_center is None or self.active_ring_completed:
            self._lock_new_ring_visit(p_est, pos_ring, dir_ring)

        center = self.active_ring_center
        n      = self.active_ring_dir
        side   = self.active_side_sign
        v_off  = self.vertical_offset
        h_off  = self.horizontal_offset

        p_near = center + side * n * h_off        + np.array([0.0, 0.0, v_off])
        p_far  = center - side * 1.25 * n * h_off + np.array([0.0, 0.0, v_off])

        if is_last_ring:
            p_target_nominal = center + np.array([0.0, 0.0, 0.1])
            self.active_ring_completed = True
        elif self.ring_phase == 'near':
            p_target_nominal = p_near
            if np.linalg.norm(p_est - p_near) <= self.near_reached_radius:
                self.ring_phase = 'far'
        else:
            p_target_nominal = p_far
            if np.linalg.norm(p_est - p_far) <= self.far_reached_radius:
                self.active_ring_completed = True

        # Define ring obstacle (only in 'near' phase)
        ring_obst = None
        if self.ring_phase == 'near':
            ring_obst = self._closest_ring_point(p_est)

        # Combined list for speed-limiting logic (dogs/drones + ring point if present)
        if ring_obst is not None:
            if pos_others is None or len(pos_others) == 0:
                pos_all_for_speed = np.array(ring_obst, ndmin=2)
            else:
                pos_all_for_speed = np.vstack([np.atleast_2d(pos_others), ring_obst])
        else:
            pos_all_for_speed = pos_others

        # Obstacle repulsion with separate gains for ring and other obstacles
        h_repel_other = self._obstacle_repulsion(p_est, pos_others, self.k_repel)
        if ring_obst is not None:
            h_repel_ring = self._obstacle_repulsion(p_est,
                                                    np.array(ring_obst, ndmin=2),
                                                    self.k_repel_ring)
        else:
            h_repel_ring = np.zeros(3)

        h_repel = h_repel_other + h_repel_ring

        # Reference position update with obstacle repulsion
        p_target = p_target_nominal + h_repel
        v_max_local, k_ref_local = self._local_speed_limits(p_est, center, pos_all_for_speed)

        e_ref = p_target - self.p_ref
        v_ref = k_ref_local * e_ref

        speed = np.linalg.norm(v_ref)
        if speed > v_max_local:
            v_ref *= v_max_local / speed

        self.p_ref = self.p_ref + self.dt * v_ref
        p_des, v_des = self.p_ref, v_ref

        # Desired state
        xdes = np.zeros(12)
        xdes[0:3] = p_des
        xdes[6:9] = v_des
        self.xdes = xdes

        # Control Law with torque and force limits
        u = -self.K @ (self.xhat - xdes)
        u[2] = np.clip(u[2], -0.16, 0.16)
        u[3] = np.clip(u[3], -22.68, 22.68)

        # Update observer
        y = self.C @ self.xhat
        self.xhat = self.xhat + self.dt * (
            self.A @ self.xhat + self.B @ u + self.L @ (pos_markers - y)
        )

        # Outputs
        tau_x, tau_y, tau_z = u[0], u[1], u[2]
        f_z = u[3] + self.f_ze

        return tau_x, tau_y, tau_z, f_z


Add the drone to the simulator.

Reset the drone (i.e., place it at a random location in the start ring).

Run simulation until `max_time` is reached or until the drone has either reached the finish ring or been disqualified.

In [54]:
# Run a single simulation with the view enabled
simulator.add_drone(Controller, 'mlutter2', 'mlutter2.png')
simulator.reset()
simulator.run(max_time=90.0, print_debug=False)
simulator.clear_drones()

def simulate_trials(n, seeds=None):
    data_list = []
    success_list = []
    finish_times = []
    rings_per_trial = []   # NEW: store rings for each trial

    for i in range(n):
        # initialize simulator with specific seed if provided
        if seeds is not None and i < len(seeds):
            simulator = ae353_drone.Simulator(seed=seeds[i])
        else:
            simulator = ae353_drone.Simulator(seed=None)

        simulator.disable_views()
        simulator.add_drone(Controller, 'mlutter2', 'mlutter2.png')
        simulator.reset()
        simulator.run(max_time=85.0, print_debug=False)

        # data for this trial
        rings_out = []
        for r in simulator.rings:
            rings_out.append({
                'p': np.array(r['p'], dtype=float).copy(),      # center (3,)
                'R': np.array(r['R'], dtype=float).copy(),      # rotation matrix (3x3)
                'radius': float(r['radius']),                   # scalar
            })
        rings_per_trial.append(rings_out)
        data_list.append(simulator.get_data('mlutter2'))
        did_it_finish, finish_time = simulator.get_result('mlutter2')[1:]
        success_list.append(did_it_finish)
        if did_it_finish:
            finish_times.append(finish_time)
        # clear drones in this simulator instance
        simulator.clear_drones()

    return data_list, success_list, finish_times, rings_per_trial


Run trials using the seeds defined above and calculate the success rate.

In [55]:
data_list, success_list, finish_times, rings_per_trial = simulate_trials(len(seeds), seeds)
success_rate = sum(success_list) / len(success_list)
print(f'Success rate: {success_rate * 100}%')

Success rate: 95.0%


Calculate the average completion time.

In [56]:
finish_times = np.asarray(finish_times)

if finish_times.size == 0:
    print("No successful runs — average completion time undefined.")
else:
    avg_time = np.mean(finish_times)
    std_time = np.std(finish_times)
    min_time = np.min(finish_times)
    max_time = np.max(finish_times)

    print(f"Average completion time: {avg_time:.2f} s")
    print(f"Std dev: {std_time:.2f} s")
    print(f"Min / Max: {min_time:.2f} / {max_time:.2f} s")


Average completion time: 73.35 s
Std dev: 2.37 s
Min / Max: 68.36 / 79.12 s


Plot the trajectories of the drone for failed trials, along with position estimates. These plots can be used to analyze failure modes.

In [57]:
import numpy as np
import matplotlib.pyplot as plt

def plot_3d_with_rings(
    data_list,
    success_list,
    rings_per_trial,
    max_success=0,
    max_failure=0,
    plot_success=False,
    plot_failure=False,
):
    label_fs = 16
    tick_fs  = 14

    fig3d = plt.figure(figsize=(8, 6))
    ax3d = fig3d.add_subplot(111, projection='3d')
    fig2d, ax2d = plt.subplots(figsize=(7, 5))

    colors = plt.cm.tab10(np.linspace(0, 1, 10))

    # ----- choose which trials go in 3D -----
    indices_3d = []
    success_count_3d = 0
    failure_count_3d = 0

    for i, success in enumerate(success_list):
        if success:
            if not plot_success or success_count_3d >= max_success:
                continue
            success_count_3d += 1
        else:
            if not plot_failure or failure_count_3d >= max_failure:
                continue
            failure_count_3d += 1
        indices_3d.append(i)

    # ----- choose which trials go in 2D (up to double the 3D caps) -----
    indices_2d = []
    success_count_2d = 0
    failure_count_2d = 0

    for i, success in enumerate(success_list):
        if success:
            if not plot_success or success_count_2d >= 2 * max_success:
                continue
            success_count_2d += 1
        else:
            if not plot_failure or failure_count_2d >= 2 * max_failure:
                continue
            failure_count_2d += 1
        indices_2d.append(i)

    # ----- plot 3D + 2D for 3D trials -----
    color_idx = 0
    for i in indices_3d:
        data = data_list[i]

        color = colors[color_idx % len(colors)]
        color_idx += 1

        x = np.array([
            data['p_x'], data['p_y'], data['p_z'],
            data['yaw'], data['pitch'], data['roll'],
            data['v_x'], data['v_y'], data['v_z'],
            data['w_x'], data['w_y'], data['w_z']
        ]).T
        xhat = np.array(data['xhat'])

        pos_actual = x[:, 0:3]
        pos_est    = xhat[:, 0:3]

        # 3D trajectories
        ax3d.plot(pos_actual[:, 0], pos_actual[:, 1], pos_actual[:, 2], color=color)
        ax3d.plot(pos_est[:, 0], pos_est[:, 1], pos_est[:, 2],
                  linestyle='--', color=color, alpha=0.7)

        ax3d.scatter(pos_actual[0, 0],  pos_actual[0, 1],  pos_actual[0, 2],
                     marker='o', color=color)
        ax3d.scatter(pos_actual[-1, 0], pos_actual[-1, 1], pos_actual[-1, 2],
                     marker='x', color=color)

        # 2D trajectories (x-y)
        ax2d.plot(pos_actual[:, 0], pos_actual[:, 1], color='red', alpha=0.9)
        ax2d.plot(pos_est[:, 0], pos_est[:, 1],
                  linestyle='--', color='red', alpha=0.5)

        ax2d.scatter(pos_actual[0, 0],  pos_actual[0, 1],
                     marker='o', color='red')
        ax2d.scatter(pos_actual[-1, 0], pos_actual[-1, 1],
                     marker='x', color='red')

        # 3D rings for this trial (same color)
        rings = rings_per_trial[i]
        for ring in rings:
            p = ring['p']
            R = ring['R']
            r = ring['radius']

            theta = np.linspace(0, 2 * np.pi, 200)
            circle_local = np.vstack([
                np.zeros_like(theta),
                r * np.cos(theta),
                r * np.sin(theta)
            ])  # (3, N)

            circle_world = p.reshape(3, 1) + R @ circle_local

            ax3d.plot(circle_world[0, :],
                      circle_world[1, :],
                      circle_world[2, :],
                      color=color, linewidth=1)

    # ----- plot extra 2D-only trials -----
    for i in indices_2d:
        if i in indices_3d:
            continue  # already plotted above

        data = data_list[i]
        x = np.array([
            data['p_x'], data['p_y'], data['p_z'],
            data['yaw'], data['pitch'], data['roll'],
            data['v_x'], data['v_y'], data['v_z'],
            data['w_x'], data['w_y'], data['w_z']
        ]).T
        xhat = np.array(data['xhat'])

        pos_actual = x[:, 0:3]
        pos_est    = xhat[:, 0:3]

        ax2d.plot(pos_actual[:, 0], pos_actual[:, 1], color='red', alpha=0.6)
        ax2d.plot(pos_est[:, 0], pos_est[:, 1],
                  linestyle='--', color='red', alpha=0.4)

        ax2d.scatter(pos_actual[0, 0],  pos_actual[0, 1],
                     marker='o', color='red')
        ax2d.scatter(pos_actual[-1, 0], pos_actual[-1, 1],
                     marker='x', color='red')

    # ----- 2D rings: use the first 3D trial if it exists -----
    if indices_3d:
        first_idx = indices_3d[0]
        rings_2d = rings_per_trial[first_idx]

        for ring in rings_2d:
            p = ring['p']
            R = ring['R']
            r = ring['radius']

            theta = np.linspace(0, 2 * np.pi, 200)
            circle_local = np.vstack([
                np.zeros_like(theta),
                r * np.cos(theta),
                r * np.sin(theta)
            ])
            circle_world = p.reshape(3, 1) + R @ circle_local

            ax2d.plot(circle_world[0, :],
                      circle_world[1, :],
                      color='k', linewidth=2, alpha=0.7)

    # ----- Axes formatting -----
    ax3d.set_xlabel('x [m]', fontsize=label_fs)
    ax3d.set_ylabel('y [m]', fontsize=label_fs)
    ax3d.set_zlabel('z [m]', fontsize=label_fs, labelpad=8)  # key line
    ax3d.set_xlim(0.0, 17.5)
    ax3d.set_ylim(-5.0, 5.0)
    ax3d.set_zlim(0.0, 6.0)
    ax3d.tick_params(axis='both', labelsize=tick_fs)
    ax3d.tick_params(axis='z', labelsize=tick_fs)

    # keep default axes position; no tight_layout on 3D
    # (optional) tweak a bit if you want more right margin:
    # fig3d.subplots_adjust(right=0.88)

    ax2d.set_xlabel('x [m]', fontsize=label_fs)
    ax2d.set_ylabel('y [m]', fontsize=label_fs)
    ax2d.set_xlim(-4.0, 20.0)
    ax2d.set_ylim(-6.0, 6.0)
    ax2d.set_aspect('equal', adjustable='box')
    ax2d.tick_params(axis='both', labelsize=tick_fs)

    fig2d.tight_layout()
    fig2d.savefig('trajectories.pdf', bbox_inches='tight')
    plt.show()

plot_3d_with_rings(
    data_list,
    success_list,
    rings_per_trial,
    max_failure=3,
    plot_failure=True,
)

<Figure size 800x600 with 1 Axes>

<Figure size 700x500 with 1 Axes>

Most failures occur near obstacles, indicating that stability around obstacles during flight may have caused failures.

Now, generate histograms of state error, state estimation error, completion time, and computation time.

In [58]:
def collect_position_errors(data_list):
    """
    Returns:
      err_act_des : 1D array of |p - p_des| over all trials and timesteps
      err_act_est : 1D array of |p - p_hat| over all trials and timesteps
    """
    err_act_des_all = []
    err_act_est_all = []

    for data in data_list:
        # actual positions (N, 3)
        p = np.vstack([data['p_x'], data['p_y'], data['p_z']]).T  # (N, 3)

        # desired and estimated states (N, 12)
        xdes = np.asarray(data['xdes'])
        xhat = np.asarray(data['xhat'])

        p_des = xdes[:, 0:3]
        p_hat = xhat[:, 0:3]

        err_act_des = np.linalg.norm(p - p_des, axis=1)
        err_act_est = np.linalg.norm(p - p_hat, axis=1)

        err_act_des_all.append(err_act_des)
        err_act_est_all.append(err_act_est)

    if len(err_act_des_all) == 0:
        return np.array([]), np.array([])

    return np.concatenate(err_act_des_all), np.concatenate(err_act_est_all)


def extract_completion_times(success_list, data_list):
    """
    Supported patterns (per element of success_list):
      1) tuple: (failed, finished, finish_time)
      2) dict:  {'failed': bool, 'finished': bool, 'finish_time': float}
      3) bool:  True => treat last time in data['t'] as completion

    Returns:
      completion_times : 1D array of finish times for successful runs
    """
    completion_times = []

    for i, s in enumerate(success_list):
        # case 1: tuple
        if isinstance(s, tuple) and len(s) == 3:
            failed, finished, finish_time = s
            if finished and not failed and (finish_time is not None):
                completion_times.append(finish_time)

        # case 2: dict
        elif isinstance(s, dict):
            failed = s.get('failed', False)
            finished = s.get('finished', False)
            finish_time = s.get('finish_time', None)
            if finished and not failed and (finish_time is not None):
                completion_times.append(finish_time)

        # case 3: plain bool
        elif isinstance(s, bool):
            if s:
                t = np.asarray(data_list[i]['t'])
                if t.size > 0:
                    completion_times.append(t[-1])

        # otherwise: ignore

    return np.asarray(completion_times)


def collect_run_times(data_list):
    """
    Assumes each data dict has key 'run_time' with per-step controller runtime.
    Returns:
      run_times : 1D array of all controller runtimes across all trials.
    """
    all_rt = []
    for data in data_list:
        if 'run_time' in data:
            all_rt.append(np.asarray(data['run_time']))
    if len(all_rt) == 0:
        return np.array([])
    return np.concatenate(all_rt)


def plot_error_histograms(
    data_list,
    bins=50,
    label_fs=16,
    tick_fs=14,
    filename='hist_errors.pdf'
):
    """
    Figure 1: histograms of |p - p_des| and |p - p_hat|.
    """
    err_act_des, err_act_est = collect_position_errors(data_list)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ax1, ax2 = axes

    # 1) |actual - desired|
    if err_act_des.size > 0:
        ax1.hist(err_act_des, bins=bins * 50)
        ax1.set_xlim(left=0, right=np.percentile(err_act_des, 97.5))
    ax1.set_xlabel(r'$||p - p_{des}||$ [m]', fontsize=label_fs)
    ax1.set_ylabel('Count', fontsize=label_fs)
    ax1.tick_params(axis='both', labelsize=tick_fs)

    # 2) |actual - estimated|
    if err_act_est.size > 0:
        ax2.hist(err_act_est, bins=bins * 20)
        ax2.set_xlim(left=0, right=np.percentile(err_act_est, 97.5))
    ax2.set_xlabel(r'$||p - \hat{p}||$ [m]', fontsize=label_fs)
    # ax2.set_ylabel('Count', fontsize=label_fs)
    ax2.tick_params(axis='both', labelsize=tick_fs)
    

    fig.tight_layout()
    fig.savefig(filename)
    return fig, axes


def plot_completion_histogram(
    data_list,
    success_list,
    label_fs=16,
    tick_fs=14,
    filename='hist_completion_times.pdf'
):
    """
    Figure 2: histogram of completion times (successful runs).
    """
    completion_times = extract_completion_times(success_list, data_list)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))

    if completion_times.size > 0:
        ax.hist(completion_times, bins=20)

    ax.set_xlabel('Time to Finish [s]', fontsize=label_fs)
    ax.set_ylabel('Count', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)

    fig.tight_layout()
    fig.savefig(filename)
    return fig, ax


def plot_runtime_histogram(
    data_list,
    bins=50,
    label_fs=16,
    tick_fs=14,
    filename='hist_run_times.pdf'
):
    """
    Figure 3: histogram of controller computation times.
    """
    run_times = collect_run_times(data_list)

    fig, ax = plt.subplots(1, 1, figsize=(5, 4))

    if run_times.size > 0:
        ax.hist(run_times, bins=bins * 20)
        ax.set_xlim(left=0, right=np.percentile(run_times, 97.5))

    ax.set_xlabel('Run Time per Step [s]', fontsize=label_fs)
    ax.set_ylabel('Count', fontsize=label_fs)
    ax.tick_params(axis='both', labelsize=tick_fs)

    # --- Scientific notation on x-axis ---
    fmt = ScalarFormatter(useMathText=True)
    fmt.set_powerlimits((-3, 3))   # use 10^k when values are small/large
    ax.xaxis.set_major_formatter(fmt)

    fig.tight_layout()
    fig.savefig(filename)
    return fig, ax


# Example usage:
fig_err, axes_err = plot_error_histograms(
    data_list,
    bins=50,
    label_fs=16,
    tick_fs=14,
    filename='hist_errors.pdf'
)

fig_comp, ax_comp = plot_completion_histogram(
    data_list,
    success_list,
    label_fs=16,
    tick_fs=14,
    filename='hist_completion_times.pdf'
)

fig_rt, ax_rt = plot_runtime_histogram(
    data_list,
    bins=50,
    label_fs=16,
    tick_fs=14,
    filename='hist_run_times.pdf'
)

plt.show()


<Figure size 1000x400 with 2 Axes>

<Figure size 500x400 with 1 Axes>

<Figure size 500x400 with 1 Axes>

The state error histogram shows that the actual state remained nearby but not exactly at the desired state for a large amount of time.

The state estimation error histogram shows that the state estimate remains close to the actual state to a high degree.

The completion time histogram shows that the drone finishes well before the target completion time on average.

The run time per step histogram shows that the run time the controller uses per time step is sufficiently low.